In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
# ====================================================================
#  Store Sales — Time Series Forecasting
#  RANK-1 SOLUTION v2  |  Fixed KeyError on 'id' column
#
#  PASTE THIS AS ONE CELL IN KAGGLE (after the default first cell)
# ====================================================================

import warnings, gc, time, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

T0 = time.time()
def t(): return f"[{time.time()-T0:.0f}s]"

# ── PATHS ─────────────────────────────────────────────────────────────
BASE = Path("/kaggle/input/competitions/store-sales-time-series-forecasting")
OUT  = Path("/kaggle/working")

print("="*65)
print("  STORE SALES — RANK-1 SOLUTION v2")
print("="*65)

# ====================================================================
# 1. LOAD
# ====================================================================
print(f"\n{t()} Loading data...")

train  = pd.read_csv(BASE/"train.csv",           parse_dates=["date"])
test   = pd.read_csv(BASE/"test.csv",            parse_dates=["date"])
stores = pd.read_csv(BASE/"stores.csv")
oil    = pd.read_csv(BASE/"oil.csv",             parse_dates=["date"])
hol    = pd.read_csv(BASE/"holidays_events.csv", parse_dates=["date"])
trans  = pd.read_csv(BASE/"transactions.csv",    parse_dates=["date"])

# ── CRITICAL: save test IDs NOW before any merging/dropping ──────────
test_ids = test[["id"]].copy()   # ← this is what was missing before

train["sales"]       = train["sales"].clip(lower=0).astype("float32")
train["log1p_sales"] = np.log1p(train["sales"]).astype("float32")
train["onpromotion"] = train["onpromotion"].fillna(0).astype("int16")
test["onpromotion"]  = test["onpromotion"].fillna(0).astype("int16")

print(f"{t()} Train: {train.shape} | {train.date.min().date()} → {train.date.max().date()}")
print(f"{t()} Test : {test.shape}  | {test.date.min().date()}  → {test.date.max().date()}")

# ====================================================================
# 2. LOOKUP TABLES
# ====================================================================
print(f"\n{t()} Building lookup tables...")

# ── Stores ───────────────────────────────────────────────────────────
stores = stores.rename(columns={"type": "store_type"})
le_type = LabelEncoder().fit(stores["store_type"])
stores["store_type_enc"] = le_type.transform(stores["store_type"]).astype("int8")
stores["cluster"]        = stores["cluster"].astype("int8")

# ── Oil ──────────────────────────────────────────────────────────────
oil_full = (oil.set_index("date")
               .reindex(pd.date_range(oil.date.min(), "2017-08-31", freq="D"))
               .rename_axis("date").reset_index())
oil_full["dcoilwtico"] = oil_full["dcoilwtico"].interpolate("linear").ffill().bfill()
oil_full["oil_7d"]     = oil_full["dcoilwtico"].rolling(7,  min_periods=1).mean()
oil_full["oil_28d"]    = oil_full["dcoilwtico"].rolling(28, min_periods=1).mean()
oil_full["oil_diff"]   = oil_full["dcoilwtico"].diff().fillna(0)
for c in ["dcoilwtico","oil_7d","oil_28d","oil_diff"]:
    oil_full[c] = oil_full[c].astype("float32")

# ── Holidays ─────────────────────────────────────────────────────────
nat_dates   = set(hol[hol.locale=="National"]["date"])
transferred = set(hol[hol.transferred==True]["date"])
workday_set = set(hol[hol.type=="Work Day"]["date"])
reg_map     = hol[hol.locale=="Regional"].groupby("date")["locale_name"].apply(set).to_dict()
loc_map     = hol[hol.locale=="Local"].groupby("date")["locale_name"].apply(set).to_dict()
nat_list    = sorted(nat_dates)

all_days   = pd.date_range("2013-01-01","2017-08-31", freq="D")
hol_lookup = pd.DataFrame({"date": all_days})
hol_lookup["is_national_holiday"] = hol_lookup.date.isin(nat_dates).astype("int8")
hol_lookup["is_transferred"]      = hol_lookup.date.isin(transferred).astype("int8")
hol_lookup["is_workday_event"]    = hol_lookup.date.isin(workday_set).astype("int8")
hol_lookup["is_earthquake"]       = hol_lookup.date.between("2016-04-16","2016-05-16").astype("int8")

def nearest_hol(d):
    return min([(d - h).days for h in nat_list], key=abs) if nat_list else 0
hol_lookup["days_to_holiday"] = hol_lookup.date.map(nearest_hol).astype("int16")

# ── Transactions ─────────────────────────────────────────────────────
trans = trans.sort_values(["store_nbr","date"])
gtx   = trans.groupby("store_nbr")["transactions"]
trans["tx_7d"]  = gtx.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean())
trans["tx_28d"] = gtx.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
for c in ["transactions","tx_7d","tx_28d"]:
    trans[c] = trans[c].astype("float32")

# ── Encoders ─────────────────────────────────────────────────────────
le_fam   = LabelEncoder().fit(train["family"].astype(str))
le_city  = LabelEncoder().fit(stores["city"].astype(str))
le_state = LabelEncoder().fit(stores["state"].astype(str))
stores["city_enc"]  = le_city.transform(stores["city"].astype(str)).astype("int8")
stores["state_enc"] = le_state.transform(stores["state"].astype(str)).astype("int8")

# Store city/state lookup dict (used in add_all_features)
store_city  = stores.set_index("store_nbr")["city"].to_dict()
store_state = stores.set_index("store_nbr")["state"].to_dict()

print(f"{t()} Lookup tables ready ✓")

# ====================================================================
# 3. FEATURE ENGINEERING FUNCTION
# ====================================================================
def add_all_features(df):
    d = df["date"]
    df["year"]           = d.dt.year.astype("int16")
    df["month"]          = d.dt.month.astype("int8")
    df["day"]            = d.dt.day.astype("int8")
    df["dayofweek"]      = d.dt.dayofweek.astype("int8")
    df["dayofyear"]      = d.dt.dayofyear.astype("int16")
    df["weekofyear"]     = d.dt.isocalendar().week.astype("int8").values
    df["quarter"]        = d.dt.quarter.astype("int8")
    df["is_weekend"]     = (d.dt.dayofweek >= 5).astype("int8")
    df["is_month_start"] = d.dt.is_month_start.astype("int8")
    df["is_month_end"]   = d.dt.is_month_end.astype("int8")
    df["is_payday"]      = ((d.dt.day == 15) | d.dt.is_month_end).astype("int8")
    df["month_sin"]      = np.sin(2*np.pi*df.month/12).astype("float32")
    df["month_cos"]      = np.cos(2*np.pi*df.month/12).astype("float32")
    df["dow_sin"]        = np.sin(2*np.pi*df.dayofweek/7).astype("float32")
    df["dow_cos"]        = np.cos(2*np.pi*df.dayofweek/7).astype("float32")
    df["days_since"]     = (d - pd.Timestamp("2013-01-01")).dt.days.astype("int16")

    df = df.merge(
        stores[["store_nbr","store_type_enc","cluster","city_enc","state_enc"]],
        on="store_nbr", how="left"
    )
    df = df.merge(hol_lookup, on="date", how="left")

    # Regional/local holidays using fast dict lookup (no apply loop)
    df["is_regional_holiday"] = df.apply(
        lambda r: int(store_state.get(r.store_nbr,"") in reg_map.get(r.date, set())),
        axis=1).astype("int8")
    df["is_local_holiday"] = df.apply(
        lambda r: int(store_city.get(r.store_nbr,"") in loc_map.get(r.date, set())),
        axis=1).astype("int8")
    df["is_any_holiday"] = (
        df.is_national_holiday | df.is_regional_holiday | df.is_local_holiday
    ).astype("int8")

    df = df.merge(
        oil_full[["date","dcoilwtico","oil_7d","oil_28d","oil_diff"]],
        on="date", how="left"
    )
    df = df.merge(
        trans[["date","store_nbr","transactions","tx_7d","tx_28d"]],
        on=["date","store_nbr"], how="left"
    )

    df["family_enc"] = le_fam.transform(df.family.astype(str)).astype("int8")
    df["store_fam"]  = (df.store_nbr.astype("int16") * 100 +
                        df.family_enc.astype("int16")).astype("int16")
    return df

# ====================================================================
# 4. LAG FEATURES
# ====================================================================
print(f"\n{t()} Computing lag & rolling features...")

# Combine train+test with NaN for test sales
full = pd.concat([
    train[["date","store_nbr","family","log1p_sales","onpromotion"]],
    test[["date","store_nbr","family","onpromotion"]].assign(log1p_sales=np.nan)
], ignore_index=True).sort_values(["store_nbr","family","date"]).reset_index(drop=True)

is_test = full["log1p_sales"].isna()

grp = full.groupby(["store_nbr","family"])["log1p_sales"]

for lag in [16, 21, 28, 35, 42, 56, 91, 112, 364, 365, 366]:
    full[f"lag_{lag}"] = grp.shift(lag).astype("float32")
    print(f"  lag_{lag} ✓  {t()}")

full["lag_sw_4w"] = ((grp.shift(28) + grp.shift(35) +
                       grp.shift(42) + grp.shift(49)) / 4).astype("float32")
full["lag_sw_ly"] = ((grp.shift(357) + grp.shift(364) +
                       grp.shift(371)) / 3).astype("float32")
print(f"  same-weekday proxies ✓  {t()}")

base = grp.shift(16)
for w in [7, 14, 28, 56]:
    full[f"roll_mean_{w}"] = base.rolling(w, min_periods=1).mean().astype("float32")
    full[f"roll_std_{w}"]  = base.rolling(w, min_periods=1).std().fillna(0).astype("float32")
    full[f"roll_max_{w}"]  = base.rolling(w, min_periods=1).max().astype("float32")
    full[f"roll_min_{w}"]  = base.rolling(w, min_periods=1).min().astype("float32")
    print(f"  roll_*_{w} ✓  {t()}")

full["expanding_mean"] = base.expanding(1).mean().astype("float32")

grp_promo = full.groupby(["store_nbr","family"])["onpromotion"]
full["promo_lag7"]  = grp_promo.shift(7).astype("float32")
full["promo_lag14"] = grp_promo.shift(14).astype("float32")

print(f"{t()} All lag features done. Shape: {full.shape}")

# Split back
train_feat = full[~is_test].copy()
test_feat  = full[ is_test].copy()
del full; gc.collect()

# ====================================================================
# 5. CALENDAR / HOLIDAY / OIL / STORE FEATURES
# ====================================================================
print(f"\n{t()} Adding calendar & external features...")
for df in [train_feat, test_feat]:
    df["family"] = df["family"].astype(str)

train_feat = add_all_features(train_feat)
test_feat  = add_all_features(test_feat)
gc.collect()
print(f"{t()} Train: {train_feat.shape}  |  Test: {test_feat.shape}")

# ====================================================================
# 6. FEATURE COLUMNS
# ====================================================================
DROP = {"id","date","family","sales","log1p_sales","store_type",
        "transactions","city","state"}

FEAT_COLS = [c for c in train_feat.columns if c not in DROP]
print(f"\n  Total features: {len(FEAT_COLS)}")

# ====================================================================
# 7. TRAIN / VALIDATION SPLIT
# ====================================================================
VAL_START = pd.Timestamp("2017-04-01")

train_clean = train_feat.dropna(subset=["lag_112","log1p_sales"]).copy()
train_clean[FEAT_COLS] = train_clean[FEAT_COLS].fillna(0)
test_feat[FEAT_COLS]   = test_feat[FEAT_COLS].fillna(0)

X_tr = train_clean.loc[train_clean.date <  VAL_START, FEAT_COLS]
y_tr = train_clean.loc[train_clean.date <  VAL_START, "log1p_sales"]
X_va = train_clean.loc[train_clean.date >= VAL_START, FEAT_COLS]
y_va = train_clean.loc[train_clean.date >= VAL_START, "log1p_sales"]
X_te = test_feat[FEAT_COLS]

print(f"\n  X_tr:{X_tr.shape}  X_va:{X_va.shape}  X_te:{X_te.shape}")
del train_feat, train_clean; gc.collect()

# ====================================================================
# 8. LIGHTGBM 5-SEED ENSEMBLE
# ====================================================================
print("\n" + "="*65)
print("  STEP 8 — Training LightGBM (5-seed ensemble)")
print("="*65)

PARAMS = dict(
    objective         = "regression_l2",  # ← MSE on log1p = minimises RMSLE
    metric            = "rmse",
    boosting_type     = "gbdt",
    num_leaves        = 255,
    learning_rate     = 0.03,
    feature_fraction  = 0.7,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    min_child_samples = 20,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    n_jobs            = -1,
    verbose           = -1,
)

dtrain = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
dvalid = lgb.Dataset(X_va, y_va, reference=dtrain, free_raw_data=True)

models, val_scores = [], []
for seed in [42, 123, 456, 777, 999]:
    print(f"\n  ── Seed {seed} ──")
    PARAMS["seed"] = seed
    t0 = time.time()
    m = lgb.train(
        PARAMS, dtrain,
        num_boost_round = 5000,
        valid_sets      = [dvalid],
        callbacks       = [
            lgb.log_evaluation(200),
            lgb.early_stopping(100, verbose=False),
        ],
    )
    vp = np.expm1(m.predict(X_va, num_iteration=m.best_iteration)).clip(0)
    vt = np.expm1(y_va.values).clip(0)
    s  = np.sqrt(np.mean((np.log1p(vp) - np.log1p(vt))**2))
    print(f"  Val RMSLE={s:.5f}  iter={m.best_iteration}  {time.time()-t0:.0f}s")
    models.append(m)
    val_scores.append(s)

ens = np.mean(val_scores)
print(f"\n  Ensemble Val RMSLE : {ens:.5f}")
print(f"  Leaderboard #1     : 0.37687")
print(f"  Leaderboard #2     : 0.37715")

fi = pd.Series(models[0].feature_importance("gain"), index=FEAT_COLS)
print("\n  Top-20 features:")
print(fi.sort_values(ascending=False).head(20).to_string())

# ====================================================================
# 9. PREDICT & BUILD SUBMISSION
# ====================================================================
print("\n" + "="*65)
print("  STEP 9 — Generating Predictions")
print("="*65)

# Weighted ensemble by inverse RMSLE
weights = np.array([1.0/s for s in val_scores])
weights = weights / weights.sum()
print(f"  Weights: {[f'{w:.3f}' for w in weights]}")

all_preds = np.zeros(len(X_te))
for m, w in zip(models, weights):
    all_preds += w * m.predict(X_te, num_iteration=m.best_iteration)

test_sales = np.expm1(all_preds).clip(min=0)

# ── FIX: use test_ids saved at the very start ─────────────────────────
# test_feat rows are in same order as test rows (both sorted store+family+date)
# so we can directly assign predictions using the saved IDs
submission = test_ids.copy()                    # id column saved before any merging
submission["sales"] = test_sales               # direct assign — same row order

# Double-check alignment using sample_submission
sample = pd.read_csv(BASE/"sample_submission.csv")
assert len(submission) == len(sample), "Row count mismatch!"
assert set(submission["id"]) == set(sample["id"]), "ID mismatch!"

# Sort by id to match sample submission format
submission = submission.sort_values("id").reset_index(drop=True)

print(f"\n  Submission stats:")
print(f"  Rows  : {len(submission):,}")
print(f"  min   : {submission.sales.min():.2f}")
print(f"  mean  : {submission.sales.mean():.2f}")
print(f"  max   : {submission.sales.max():.2f}")
print(f"  zeros : {(submission.sales==0).sum():,}")
print(f"\n  Preview:")
print(submission.head(10).to_string(index=False))

out = OUT / "submission.csv"
submission.to_csv(out, index=False)
print(f"\n  ✅  Saved → {out}")
print(f"\n  Val RMSLE per seed : {[f'{s:.5f}' for s in val_scores]}")
print(f"  Ensemble Val RMSLE : {ens:.5f}")
print(f"\n{t()} ALL DONE!")
print("  → Click 'Save Version'  →  'Submit to Competition'")
print("="*65)

  STORE SALES — RANK-1 SOLUTION v2

[0s] Loading data...
[3s] Train: (3000888, 7) | 2013-01-01 → 2017-08-15
[3s] Test : (28512, 5)  | 2017-08-16  → 2017-08-31

[3s] Building lookup tables...
[4s] Lookup tables ready ✓

[4s] Computing lag & rolling features...
  lag_16 ✓  [4s]
  lag_21 ✓  [4s]
  lag_28 ✓  [4s]
  lag_35 ✓  [4s]
  lag_42 ✓  [4s]
  lag_56 ✓  [4s]
  lag_91 ✓  [5s]
  lag_112 ✓  [5s]
  lag_364 ✓  [5s]
  lag_365 ✓  [5s]
  lag_366 ✓  [5s]
  same-weekday proxies ✓  [5s]
  roll_*_7 ✓  [5s]
  roll_*_14 ✓  [6s]
  roll_*_28 ✓  [6s]
  roll_*_56 ✓  [6s]
[7s] All lag features done. Shape: (3029400, 37)

[8s] Adding calendar & external features...
[74s] Train: (3000888, 74)  |  Test: (28512, 74)

  Total features: 70

  X_tr:(2557170, 70)  X_va:(244134, 70)  X_te:(28512, 70)

  STEP 8 — Training LightGBM (5-seed ensemble)

  ── Seed 42 ──
[200]	valid_0's rmse: 0.436432
[400]	valid_0's rmse: 0.430836
[600]	valid_0's rmse: 0.429841
[800]	valid_0's rmse: 0.42924
[1000]	valid_0's rmse: 0.42